In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from dotenv import dotenv_values
from IPython.display import clear_output
from omegaconf import OmegaConf
from pathlib import Path

from annotate import display_fixed_image
from beyond_hate.analysis.utils import compute_pairwise_agreement

cfg = OmegaConf.load("../../config/default.yaml")

env_values = dotenv_values()
hf_token = env_values["HF_TOKEN"]

# Resolve paths
project_root = Path("../..").resolve()
labels_file = project_root / cfg.data.paths.labels_file
hf_data = project_root / cfg.data.paths.hf

In [ ]:
df = pd.DataFrame()
splits = ['dev_seen', 'dev_unseen', 'test_seen', 'test_unseen', 'train']
for split in splits:
    df_tmp = pd.read_json(f'{hf_data}/{split}.jsonl', lines=True)
    df_tmp['split'] = split
    df = pd.concat([df, df_tmp])

# Drop images that are not found in the local HF dataset and duplicates
hf_images = os.listdir(hf_data / "img")
df['image_found'] = df['img'].str.lstrip('img/').isin(hf_images)
df = df[df['image_found'] == True]
df = df.drop_duplicates(subset=['img'], keep='first')

# Load relevant ids
with open(project_root / cfg.data.paths.base / "images_to_annotate.txt", "r") as f:
    lines = f.readlines()
    relevant_ids = [int(line.strip()) for line in lines]

# Load annotations and keep only relevant ones
annotations = pd.read_json(labels_file, lines=True,
                           dtype={"label_hateful": int,
                                  "label_incivility": str,
                                  "label_intolerance": str})
annotations = annotations[annotations["id"].isin(relevant_ids)]

# Validate
## Check if all images where annotated by each annotator
annotator_counts = annotations.groupby("annotator")["id"].nunique()
expected_count = len(relevant_ids)
for annotator_id, count in annotator_counts.items():
    if count != expected_count:
        warnings.warn(f"Annotator {annotator_id} annotated {count} images, expected {expected_count}")

## Check if 0 label and other labels are mutually exclusive
def has_zero_with_others(label_str):
    labels = label_str.split(',')
    return '0' in labels and len(labels) > 1

incivility_violations = annotations['label_incivility'].apply(has_zero_with_others).sum()
intolerance_violations = annotations['label_intolerance'].apply(has_zero_with_others).sum()

if incivility_violations > 0:
    warnings.warn(f"Found {incivility_violations} incivility labels with '0' mixed with others")
if intolerance_violations > 0:
    warnings.warn(f"Found {intolerance_violations} intolerance labels with '0' mixed with others")

# Drop duplicates by annotator (keep last)
annotations = annotations.drop_duplicates(subset=["annotator", "id"], keep="last")

# Binarize labels for agreement calculation
annotations.loc[:, "label_incivility_bin"]  = annotations["label_incivility"].apply(lambda x: 1 if "0" not in x else 0)
annotations.loc[:,"label_intolerance_bin"] = annotations["label_intolerance"].apply(lambda x: 1 if "0" not in x else 0)

In [ ]:
## Compute inter-annotator agreement
print("INTER-ANNOTATOR AGREEMENT")
print("=" * 50)

annotators = annotations["annotator"].unique()
print(f"Number of annotators: {len(annotators)}")
print(f"Annotators: {list(annotators)}")

# Hateful labels
print("\nHATEFUL LABELS:")
hate_agreements, hate_kappas = compute_pairwise_agreement(annotations, "label_hateful")
if hate_agreements:
    print(f"Average agreement: {np.mean(hate_agreements):.3f} ± {np.std(hate_agreements):.3f}")
    print(f"Average Kappa: {np.mean(hate_kappas):.3f} ± {np.std(hate_kappas):.3f}")

# Incivility labels  
print("\nINCIVILITY LABELS:")
inciv_agreements, inciv_kappas = compute_pairwise_agreement(annotations, "label_incivility_bin")
if inciv_agreements:
    print(f"Average agreement: {np.mean(inciv_agreements):.3f} ± {np.std(inciv_agreements):.3f}")
    print(f"Average Kappa: {np.mean(inciv_kappas):.3f} ± {np.std(inciv_kappas):.3f}")

# Intolerance labels
print("\nINTOLERANCE LABELS:")
intol_agreements, intol_kappas = compute_pairwise_agreement(annotations, "label_intolerance_bin")
if intol_agreements:
    print(f"Average agreement: {np.mean(intol_agreements):.3f} ± {np.std(intol_agreements):.3f}")
    print(f"Average Kappa: {np.mean(intol_kappas):.3f} ± {np.std(intol_kappas):.3f}")

print("\n" + "=" * 50)

In [ ]:
# Save images to re-annotate

# Keep gold-standard annotators
senior_annotators = ["Annotator_1", "Annotator_2"]
senior_annotations = annotations[annotations["annotator"].isin(senior_annotators)]

disagreement_mask = (
    senior_annotations.groupby("id")[["label_incivility_bin", "label_intolerance_bin"]]
    .apply(lambda x: (x["label_incivility_bin"].nunique() > 1) or (x["label_intolerance_bin"].nunique() > 1))
)

disagreed_images = senior_annotations[senior_annotations["id"].isin(disagreement_mask[disagreement_mask].index)]
print(f"Number of images with disagreement among senior annotators: {disagreed_images['id'].nunique()}")
print(f"Share of images with disagreement among senior annotators: {disagreed_images['id'].nunique() / senior_annotations['id'].nunique():.2%}")

# Save ids of images to re-annotate
with open(project_root / cfg.data.paths.base / "images_to_reannotate.txt", "w") as f:
    for img_id in disagreed_images["id"].unique():
        f.write(f"{img_id}\n")

In [ ]:
# Gather examples for calibration session
gold_set = annotations[~annotations["id"].isin(disagreement_mask[disagreement_mask].index)]
print(f"Gold set: {gold_set['id'].nunique()}")
print(f"Gold set: {gold_set['id'].nunique() / annotations['id'].nunique():.2%}")

# Remove images where junior annotator agrees with senior annotators
disagreement_mask = (
    gold_set.groupby("id")[["label_incivility_bin", "label_intolerance_bin"]]
    .apply(lambda x: (x["label_incivility_bin"].nunique() > 1) or (x["label_intolerance_bin"].nunique() > 1))
)

disagreed_images = gold_set[gold_set["id"].isin(disagreement_mask[disagreement_mask].index)]
print(f"Gold set with disagreement with junior annotator: {disagreed_images['id'].nunique() / annotations['id'].nunique():.2%}")

In [ ]:
# Sample n images for each combination of labels from senior annotators
n = 3

# Get only senior annotators' labels from disagreed_images
disagreed_senior = disagreed_images[disagreed_images["annotator"].isin(senior_annotators)]

# Group by combination of labels and sample n images from each group
sampled_images = (
    disagreed_senior
    .groupby(["label_incivility_bin", "label_intolerance_bin", "label_hateful"])
    .apply(lambda x: x.sample(n=min(n, len(x)), random_state=42))
    .reset_index(drop=True)
)

sampled_ids = sampled_images["id"].unique()

print(f"Total sampled: {len(sampled_ids)}")

In [ ]:
with open(project_root / cfg.data.paths.base / "images_to_annotate.txt", "r") as f:
    lines = f.readlines()
    images_to_annotate = [int(line.strip()) for line in lines]

In [ ]:
df_not_annotated = df[~df['id'].isin(images_to_annotate)]
print(f"Share of images not yet annotated: {len(df_not_annotated) / len(df):.2%}")

# Sample 3 images per split and 3 per label
additional_ids = df_not_annotated.groupby(["split", "label"], group_keys=False).apply(lambda x: x.sample(n=min(n, len(x)), random_state=42))["id"].unique()
print(f"Additional sampled: {len(additional_ids)}")

# Exclude some examples which will be used before the re-annotation
manually_selected_ids = [64071, 43092, 2783, 30417, 38754, 9516, 24560, 2957, 36104, 98103, 45396, 9384]  # Example IDs to exclude

In [ ]:
# Show manually selected images
for row in df[df["id"].isin(manually_selected_ids)].itertuples():
    clear_output(wait=True)
    display_fixed_image(hf_data / row.img)
    user_input = input()
    if user_input.lower() == 'q':
        break

In [ ]:
calibration_ids = np.concatenate([sampled_ids, additional_ids])
calibration_ids = [img_id for img_id in calibration_ids if img_id not in manually_selected_ids]
print(f"Total manually selected images for discussion: {len(manually_selected_ids)}")
print(f"Total images for calibration session: {len(calibration_ids)}")

with open(project_root / cfg.data.paths.base / "images_to_calibrate.txt", "w") as f:
    for img_id in calibration_ids:
        f.write(f"{img_id}\n")